# The Free-Coverage Diagnostic — a real-world walkthrough

**Goal of this notebook:** show, on a *real dataset*, how to decide whether wrapping a
predict-then-optimize system in conformal prediction is **free** (it never changes the decision)
or **costly** (it changes decisions and adds cost) — *before* you deploy it.

### Background in one minute

Many operational decisions are **predict-then-optimize (PtO)**: an ML model predicts costs, and an
optimizer (a linear program) turns those predictions into a decision — which generators to run,
which routes to dispatch, how to staff wards, where to place inventory or capital.

Point predictions ignore uncertainty. **Online conformal prediction** fixes that: it wraps each
prediction in a calibrated uncertainty set that tracks a target coverage rate (e.g. 90%) with no
distributional assumptions, even under drift. The optimizer then solves a **robust** version that
hedges against costs anywhere in that set.

But hedging can cost money — the optimizer may switch to a more expensive, "safer" decision. We
call that premium the **Price of Coverage (PoC)**. The key insight behind this diagnostic:

> **On many problems the Price of Coverage is *zero*.** When the calibrated uncertainty is smaller
> than the cost gap to the next-best decision, the robust optimum *equals* the nominal optimum:
> coverage changes nothing about the decision, so you get a 90%-coverage *certificate for free*.

`FreeCoverageDiagnostic` measures exactly this on your own predictor and LP and returns a
**free / critical / costly** verdict. This notebook runs it end-to-end on real data.

*Method: "When Is Conformal Coverage Free? Switching Thresholds for Predict-then-Optimize" (COPA 2026).*

### Where this matters (and what we use here)

The verdict is problem-dependent, and the paper's real benchmarks span both regimes:

| Real problem | Decision | Typical verdict |
|---|---|---|
| Energy economic dispatch (few generators) | which generators to commit | **free** — wide cost gaps |
| Vaccine / antiviral allocation (few regions) | how to split supply | **free / critical** |
| City-scale taxi routing (thousands of edges) | which route to take | **costly** — many near-tied paths |

Those datasets are large and licensed, so **this notebook uses a fully self-contained public
dataset** — sklearn's California Housing — as a runnable stand-in for a **budget-allocation**
decision. The *mechanics and interpretation are identical* to the problems above; only the data
differs. Everything here runs offline in a few seconds with `pip install "conformal-ops[examples]"`.

## 1. The decision problem

A real-estate fund has a fixed capital budget each period and a shortlist of `d` candidate
districts. An ML model predicts each district's price from its features; the fund wants to acquire
exposure at **minimum total predicted price** subject to the budget and per-district caps — a small
linear program. Prices are uncertain, so we wrap the predictions in conformal sets and ask:

> **Does hedging against price-prediction error change *which districts we buy*?**

If not, the 90%-coverage certificate is free. If it does, we are paying a Price of Coverage.

## 2. Data and a real predictor

California Housing: 20,640 districts, 8 features, target = median house value (in \$100k). We fit a
gradient-boosting model on the first 70% and evaluate on the rest — a realistically noisy predictor
(the noise is what conformal prediction calibrates).

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

from conformal_ops import FreeCoverageDiagnostic

X, y = fetch_california_housing(return_X_y=True)   # target: median house value ($100k)
n_train = int(0.7 * len(X))
X_tr, X_te = X[:n_train], X[n_train:]
y_tr, y_te = y[:n_train], y[n_train:]

model = GradientBoostingRegressor(
    n_estimators=200, max_depth=5, learning_rate=0.1, subsample=0.8, random_state=42
)
model.fit(X_tr, y_tr)                # trained ONCE (inductive); never refit online
yhat_te = model.predict(X_te)        # fixed predictions on the held-out stream
print(f"train districts: {len(y_tr):,}   test districts: {len(y_te):,}")
print(f"test R^2 = {r2_score(y_te, yhat_te):.3f}   (a realistically noisy predictor)")

### Map this to *your* problem

Everything below needs only three things — swap in yours:

| This notebook | Your deployment |
|---|---|
| `yhat_te` — predicted district prices | your model's cost predictions, one vector per period |
| `y_te` — realized prices | the true costs, revealed *after* you decide |
| the budget LP (`A_eq`, `b_eq`, `bounds`) | your optimizer's constraints (network flow, assignment, dispatch, …) |

The diagnostic treats the predictor as a black box and the optimizer as any `scipy.linprog` LP.

## 3. From predictions to a decision stream

Each **round** is a period in which the fund chooses among `d` candidate districts. We slice the
test set into rounds of `d` districts.

**One rule matters for a fair diagnostic** (a classic bug otherwise): the predicted and true cost
vectors must go through the **same** transformation. We normalize both with the *same*
train-derived min/max, so `c_pred` and `c_true` are directly comparable.

In [ ]:
# SAME transform for predicted AND true costs (train-derived) -> fair, comparable costs.
vmin, vmax = y_tr.min(), y_tr.max()
def norm(v):
    return 0.5 + (v - vmin) / (vmax - vmin + 1e-8)

def build_stream(pred, true, d):
    n = len(true) // d
    Cp = norm(pred[: n * d]).reshape(n, d)   # (rounds, d) predicted cost
    Ct = norm(true[: n * d]).reshape(n, d)   # (rounds, d) true cost (aligned per district)
    return Cp, Ct

# Selection LP: fund the cheapest ~1/4 of the d districts. Blocks may go to zero,
# so the FUNDED SET can genuinely change from round to round.
def selection_lp(d):
    return dict(A_eq=np.ones((1, d)), b_eq=np.array([max(1.0, d / 4.0)]),
                bounds=[(0.0, 1.0)] * d)

d = 10
Cp, Ct = build_stream(yhat_te, y_te, d)
print(f"{Cp.shape[0]} rounds of d={d} districts each")

## 4. Run the diagnostic

A single call runs a short online-conformal calibration pilot over the stream and returns the verdict.

In [ ]:
report = FreeCoverageDiagnostic(alpha=0.10).run(Cp, Ct, **selection_lp(d))
print(report)

### Reading the report — field by field

- **regime** — the headline verdict: `free`, `critical`, or `costly`.
- **decision-neutral (committed set)** — fraction of rounds on which coverage leaves the **funded
  set of districts** unchanged (the robust and nominal optima activate the same districts). This is
  usually the decision you actually care about.
- **decision-neutral (full vector)** — fraction on which the **exact allocation amounts** are
  unchanged. Always ≤ the committed-set number (identical vectors are trivially set-identical).
- **coverage** — realized joint coverage of the conformal sets over the pilot. It should sit near
  `1 − α = 90%`; this is your sanity check that calibration is working.
- **q\*, sigma_q** — the mean and standard deviation of the online conformal quantile over the
  pilot: the *size* of the uncertainty and how much it fluctuates.

On this problem the verdict is **costly**: with many similarly-priced districts, the conformal radii
frequently reshuffle which districts are cheapest, so the funded set moves. Coverage here is *not*
free — you would pay a Price of Coverage, and might reach for `DICA` to reduce it.

## 5. Committed set vs full vector — the same coverage, two answers

Whether coverage is "free" can depend on *which* decision you mean. This is not hair-splitting — it
is exactly the distinction between **which generators are committed** (a stable, discrete decision)
and **their exact MW output** (which shifts constantly) in energy dispatch.

We contrast the selection LP (funded set can change) with a **bounded** budget LP that forces every
district to receive at least a floor allocation — there the funded *set* is fixed **by
construction** (always all `d`), even though the exact amounts still move.

In [ ]:
def bounded_lp(d):
    # every district floored at 0.1 -> support is ALWAYS all d districts
    return dict(A_eq=np.ones((1, d)), b_eq=np.array([0.6 * d]), bounds=[(0.1, 1.0)] * d)

for label, lp in (("selection (lb=0)", selection_lp), ("bounded (lb=0.1)", bounded_lp)):
    rep = FreeCoverageDiagnostic(alpha=0.10).run(Cp, Ct, **lp(d))
    print(f"{label:>18}:  committed={rep.neutral_frac_committed:5.1%}   "
          f"vector={rep.neutral_frac_vector:5.1%}   regime={rep.regime}")

The bounded LP reports **committed = 100%** — the floor keeps every district funded, so coverage
never changes *whether* a district is in the portfolio, only *how much*. Its vector neutrality is
low, because the amounts do move.

**Takeaway:** report *both* granularities and pick the one that matches your real decision. If the
decision is "which districts / which generators / which routes", use the committed-set number. If it
is "exactly how much of each", use the vector number. The diagnostic gives you both so you cannot be
misled by a metric that is trivially 100% (as the committed-set number is whenever every variable is
floored above zero).

## 6. Dimension dependence — bigger decisions are less free

As the number of candidate districts `d` grows, more of them sit at nearly the same price, so the
cost gap to the next-best option shrinks and coverage flips the funded set more often. This is the
paper's result that the switching threshold scales as **κ\* = O(1/K_det)** (more near-optimal
alternatives ⇒ smaller free region) — here it is, on real data. It is the same reason energy
dispatch (few generators) is free while city-scale routing (thousands of near-tied paths) is costly.

In [ ]:
print(" d | rounds | committed | vector | coverage | regime")
sweep = []
for d_ in (5, 10, 20, 40):
    Cp_, Ct_ = build_stream(yhat_te, y_te, d_)
    r = FreeCoverageDiagnostic(alpha=0.10).run(Cp_, Ct_, **selection_lp(d_))
    sweep.append((d_, r.neutral_frac_committed))
    print(f"{d_:>2} | {r.n_rounds:>6} | {r.neutral_frac_committed:8.1%} | "
          f"{r.neutral_frac_vector:6.1%} | {r.coverage:7.1%} | {r.regime}")

In [ ]:
# Optional plot (needs matplotlib: pip install "conformal-ops[examples]")
try:
    import matplotlib.pyplot as plt
    ds, frac = zip(*sweep)
    plt.figure(figsize=(6, 4))
    plt.plot(ds, [f * 100 for f in frac], "o-", color="steelblue")
    plt.axhline(95, ls="--", c="green", lw=1, label="free threshold (95%)")
    plt.axhline(50, ls="--", c="firebrick", lw=1, label="costly threshold (50%)")
    plt.xlabel("candidate districts per round  (d)")
    plt.ylabel("committed-set decision-neutral (%)")
    plt.title("Coverage becomes less free as the decision grows")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
except ImportError:
    print("(install matplotlib to see the plot)")

## 7. The analytic switching threshold κ\* (optional)

The empirical verdict above needs no problem structure — it just measures decision changes. If you
*can* enumerate the competing decisions (the K-shortest paths in routing, the alternative vertices
in a small LP), the diagnostic also computes the **analytic** switching threshold κ\* and a
**safety margin** `m = (κ* − q*) / σ_q`: how many standard deviations of headroom the conformal
quantile has before it would flip the decision. `m > 2` ⇒ free, `m < −2` ⇒ costly.

We show it on a **single-pick** version (choose the one cheapest district — a totally-unimodular
selection, like a shortest path). The competing vertices are simply the other choices.

In [ ]:
d_pick = 8
Cp8, Ct8 = build_stream(yhat_te, y_te, d_pick)
single_pick_lp = dict(A_eq=np.ones((1, d_pick)), b_eq=np.array([1.0]),
                      bounds=[(0.0, 1.0)] * d_pick)   # pick exactly one district

def competitors(c_rep, x_nom):
    star = int(np.argmax(x_nom))                       # the chosen district
    return [np.eye(d_pick)[k] for k in range(d_pick) if k != star]

rep = FreeCoverageDiagnostic(alpha=0.10).run(
    Cp8, Ct8, competitors=competitors, **single_pick_lp
)
print(rep)

The analytic margin here is **negative** (`basis = margin`, `regime = costly`) — the conformal
quantile comfortably exceeds the cost gap to the next-best district — and it **agrees** with the
empirically measured neutral fraction on the same data. Two independent routes (measure decisions
directly vs. compare the calibrated radius to the structural threshold) reaching the same verdict is
exactly the cross-check you want before trusting a deployment decision.

## Takeaway

Run `FreeCoverageDiagnostic` on your **own** predictor and LP *before* deploying conformal hedging:

- **free** → coverage is a zero-cost certificate; deploy the 90%-coverage monitoring with confidence.
- **costly** → budget the premium, or **reduce** it with `DICA` (`from conformal_ops import DICA`).
- **critical** → borderline; monitor coverage closely and re-check as the data drifts.

> `FreeCoverageDiagnostic` tells you *whether* you have a Price of Coverage; `DICA` *reduces* it.

**Honest caveats.**
- California Housing is *cross-sectional*, so the "stream of rounds" here is a stylized batching to
  illustrate the online mechanics. On a real deployment, feed your genuine **time-ordered** stream so
  the online calibration sees the drift it is designed for.
- **Committed-set neutrality is only informative when the funded set *can* change** (a selection /
  `lb = 0` LP). With a positive floor on every variable it is trivially 100% — read the vector number
  instead.
- The analytic κ\* path needs a problem-specific **competitor oracle**; the empirical path (which
  needs none) is the robust default and works for any LP.